# 02 - Phase 2 EDA: Tasmanian area x day feature table

Exploration only. All logic lives in `pipeline/features/`. Rebuild the input with
`uv run python -m pipeline.features.build`.

**Rule for this notebook:** every *choice* (baseline, `MIN_VALID_PIXELS`) uses **train and
validation only**. Test rows (2024+) appear only in data-coverage counts, never in label
rates or anomaly levels, so the final evaluation stays untouched.

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from pipeline.features import target as T
from pipeline.features.build import OUT_PATH
from pipeline.features.chl import OUT_PATH as CHL_PATH

BLUE, ORANGE, MUTED = "#2a78d6", "#eb6834", "#52514e"
plt.rcParams.update(
    {
        "figure.dpi": 110,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.color": "#e6e5e1",
        "grid.linewidth": 0.6,
        "axes.edgecolor": MUTED,
        "lines.linewidth": 2,
        "font.size": 9,
    }
)
df = pd.read_parquet(OUT_PATH)
dev = df[df.split.isin(["train", "val"])]  # used for all choices
print(df.shape, df.date.min().date(), df.date.max().date(), df.area_id.nunique(), "areas")
print(df.split.value_counts().sort_index().to_string())

(219507, 61) 2016-04-25 2026-11-09 57 areas
split
test      59508
train    139194
val       20805


## 1. Missing data: valid chlorophyll days and labelled issue dates

In [2]:
df["valid_day"] = df.n_valid >= T.MIN_VALID_PIXELS
by_year = df.groupby(df.date.dt.year).agg(
    valid_day=("valid_day", "mean"),
    labelled_fixed=("y_fixed", lambda s: s.notna().mean()),
    labelled_trailing=("y_trailing", lambda s: s.notna().mean()),
)
d2 = dev.assign(valid_day=dev.n_valid >= T.MIN_VALID_PIXELS)
by_month = d2.groupby(d2.date.dt.month).agg(
    valid_day=("valid_day", "mean"), labelled_fixed=("y_fixed", lambda s: s.notna().mean())
)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.2), sharey=True)
for col, c, lab in [
    ("valid_day", BLUE, "valid chl day (>= 5 px)"),
    ("labelled_fixed", ORANGE, "labelled issue date (fixed)"),
]:
    ax[0].plot(by_year.index, by_year[col], color=c, marker="o", ms=4, label=lab)
    ax[1].plot(by_month.index, by_month[col], color=c, marker="o", ms=4, label=lab)
ax[0].axvline(2021.5, color=MUTED, ls="--", lw=1)
ax[0].text(2021.6, 0.05, "my -> myint", color=MUTED)
ax[0].set_title("Share of area-days, by year (all splits)")
ax[1].set_title("By month (train + val)")
ax[0].set_ylim(0, 1)
ax[1].legend(frameon=False, loc="lower left")
plt.tight_layout()
plt.show()
print(by_year.round(3).to_string())

      valid_day  labelled_fixed  labelled_trailing
date                                              
2016      0.194           0.408              0.000
2017      0.234           0.521              0.065
2018      0.238           0.520              0.387
2019      0.373           0.747              0.699
2020      0.322           0.687              0.681
2021      0.366           0.721              0.717
2022      0.326           0.687              0.684
2023      0.363           0.748              0.746
2024      0.373           0.747              0.744
2025      0.391           0.781              0.778
2026      0.405           0.776              0.772


Winter (May-Aug) has far fewer usable days because of low sun angle and cloud. The 2016 and
2026 rows are partial years (OLCI starts 2016-04-25; the latest data is early November 2026).
The trailing baseline needs earlier years, so it labels nothing in 2016 and less in 2017-2018
(see the table).

## 2. `MIN_VALID_PIXELS` sensitivity (train only)

In [3]:
chl = pd.read_parquet(CHL_PATH)
tr = chl[chl.date <= T.TRAIN_END].sort_values(["area_id", "date"]).copy()
tr["log"] = np.log10(tr.chl.where(tr.chl > 0))
gap = tr.date.groupby(tr.area_id).diff().dt.days
rows = []
for k in [1, 3, 5, 10, 20, 40]:
    x = tr.log.where(tr.n_valid >= k)
    # noise proxy: |change| between valid days exactly 1 day apart (real change is slower)
    d1 = (x - x.groupby(tr.area_id).shift(1)).abs()[gap == 1].dropna()
    rows.append(
        {
            "min_px": k,
            "valid_share": x.notna().mean(),
            "median_abs_dlog_1day": d1.median(),
            "p90_abs_dlog_1day": d1.quantile(0.9),
        }
    )
sens = pd.DataFrame(rows).set_index("min_px")
print(sens.round(4).to_string())

        valid_share  median_abs_dlog_1day  p90_abs_dlog_1day
min_px                                                      
1            0.3365                0.1766             0.6790
3            0.3187                0.1678             0.6287
5            0.2981                0.1575             0.5803
10           0.2555                0.1409             0.5093
20           0.1821                0.1229             0.4410
40           0.0832                0.0917             0.3564


Read down the rows: a higher minimum keeps fewer days, and day-to-day noise should fall if
small cloud-edge samples are unreliable. A good minimum sits where noise has mostly stopped
falling but coverage hasn't yet dropped steeply.

## 3. Seasonal cycle of log10 chl by growing area (train)

In [4]:
t = df[df.split == "train"]
clim = t.groupby(["growing_area", t.date.dt.month]).log_chl.median().unstack()
clim = clim.loc[clim.mean(axis=1).sort_values().index]
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(clim.values, aspect="auto", cmap="Blues")
ax.set_xticks(range(12), list("JFMAMJJASOND"))
ax.set_yticks(range(len(clim)), clim.index)
ax.grid(False)
fig.colorbar(im, ax=ax, label="median log10 chl (mg m-3)")
ax.set_title("Monthly median log10 chl, training years (rows sorted by annual level)")
plt.tight_layout()
plt.show()

The per-area x month baseline removes both the level differences between areas (estuaries vs
open coast) and the seasonal cycle. That is why the target uses anomalies.

## 4. Class balance: train and validation, both baselines

In [5]:
rate = dev.groupby("split")[["y_fixed", "y_trailing"]].agg(["mean", "count"])
print(rate.round(3).to_string())
yr = dev.groupby(dev.date.dt.year)[["y_fixed", "y_trailing"]].mean()
fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
ax[0].plot(yr.index, yr.y_fixed, color=BLUE, marker="o", ms=4, label="fixed (2016-2022)")
ax[0].plot(yr.index, yr.y_trailing, color=ORANGE, marker="o", ms=4, label="trailing (3 yr)")
ax[0].axhline(0.10, color=MUTED, ls=":", lw=1)
ax[0].text(2016.1, 0.105, "design rate 10%", color=MUTED)
ax[0].axvspan(2022.5, 2023.5, color="#f0efec")
ax[0].text(2022.62, 0.135, "val", color=MUTED)
ax[0].set_title("Positive rate by year")
ax[0].legend(frameon=False)
v = dev[dev.split == "val"].groupby("area_id")[["y_fixed", "y_trailing"]].mean()
bins = np.linspace(0, max(0.5, float(v.max().max())), 21)
ax[1].hist(v.y_fixed, bins=bins, color=BLUE, alpha=0.8, label="fixed", rwidth=0.9)
ax[1].hist(v.y_trailing, bins=bins, color=ORANGE, alpha=0.6, label="trailing", rwidth=0.9)
ax[1].set_title("Validation positive rate per area")
ax[1].set_xlabel("positive rate")
ax[1].legend(frameon=False)
plt.tight_layout()
plt.show()
for c in ["y_fixed", "y_trailing"]:
    print(
        c,
        "val per-area rate:",
        v[c].describe()[["min", "50%", "max"]].round(3).to_dict(),
        "| areas with zero val positives:",
        int((v[c] == 0).sum()),
    )

      y_fixed        y_trailing       
         mean  count       mean  count
split                                 
train   0.098  86678      0.097  67301
val     0.138  15572      0.130  15526


y_fixed val per-area rate: {'min': 0.0, '50%': 0.135, 'max': 0.271} | areas with zero val positives: 1
y_trailing val per-area rate: {'min': 0.024, '50%': 0.115, 'max': 0.38} | areas with zero val positives: 0


## 5. Fixed vs trailing baseline (validation only)

Criteria fixed **before** looking: (a) the validation positive rate should be close to the 10%
the threshold was designed for (a big shift means the baseline no longer describes "normal");
(b) per-area rates should be reasonably even, with few areas having no positives at all;
(c) the two labels should still pick out the same events, so check how far they agree.
Predictive skill is *not* used to pick the label, because a label that is easier to predict is
not necessarily a better description of blooms.

In [6]:
vv = dev[dev.split == "val"].dropna(subset=["y_fixed", "y_trailing"])
agree = (vv.y_fixed == vv.y_trailing).mean()
pos_f, pos_t = vv.y_fixed == 1, vv.y_trailing == 1
jac = (pos_f & pos_t).sum() / max(1, (pos_f | pos_t).sum())
print(
    f"val rows with both labels: {len(vv):,}; agreement {agree:.3f}; Jaccard of positives {jac:.3f}"
)
print(
    "val median daily anomaly - fixed:",
    round(vv.anomaly_fixed.median(), 3),
    " trailing:",
    round(vv.anomaly_trailing.median(), 3),
)
m = dev.groupby(dev.date.dt.to_period("Q"))[["anomaly_fixed", "anomaly_trailing"]].median()
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(m.index.to_timestamp(), m.anomaly_fixed, color=BLUE, label="fixed")
ax.plot(m.index.to_timestamp(), m.anomaly_trailing, color=ORANGE, label="trailing")
ax.axhline(0, color=MUTED, lw=1)
ax.axvline(pd.Timestamp("2022-01-01"), color=MUTED, ls="--", lw=1)
ax.set_title("Quarterly median daily anomaly, all areas (train + val)")
ax.set_ylabel("log10")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

val rows with both labels: 15,526; agreement 0.918; Jaccard of positives 0.532
val median daily anomaly - fixed: 0.013  trailing: -0.008


## 6. Drivers: coverage and simple association with the target (train only)

In [7]:
meta = {"area_id", "date", "split", "match_key", "growing_area", "processing", "valid_day"}
targ = [c for c in df.columns if c.endswith(("_fixed", "_trailing"))] + ["log_chl", "n_valid"]
feat = [c for c in df.columns if c not in meta | set(targ)]
t = df[df.split == "train"]
tab = pd.DataFrame(
    {
        "missing_share": t[feat].isna().mean(),
        # Spearman = Pearson on ranks (avoids a scipy dependency)
        "spearman_vs_fwd_mean_fixed": t[feat].rank().corrwith(t.fwd_mean_fixed.rank()),
    }
).sort_values("spearman_vs_fwd_mean_fixed", key=abs, ascending=False)
print(len(feat), "features")
print(tab.round(3).to_string())

43 features
                    missing_share  spearman_vs_fwd_mean_fixed
chl_anom_mean14             0.044                       0.183
chl_anom_lag2               0.702                       0.170
rain_past30                 0.000                       0.164
chl_anom_mean7              0.146                       0.163
chl_anom_lag3               0.703                       0.157
rain_past14                 0.000                       0.142
chl_anom_lag4               0.703                       0.138
chl_anom_lag5               0.703                       0.137
chl_anom_last               0.044                       0.137
chl_anom_lag7               0.703                       0.127
chl_anom_max14              0.044                       0.127
chl_anom_lag6               0.703                       0.126
chl_anom_lag8               0.703                       0.118
rain_past7                  0.000                       0.104
chl_level_last              0.044                       0.

Rank correlation is only a sanity check: it misses interactions and non-linear effects, which
the gradient-boosting model (Phase 3) can capture. Recent chlorophyll anomaly should dominate
(persistence), which is why persistence is a required baseline to beat.

## 7. Harmful-species events at Maria Island (context)

Maria Island (MAI) is Tasmania's only NRS station. The 2025-26 *Karenia* bloom is an SA event,
and SA areas are out of scope until PIRSA replies, so its case study waits for SA. This shows
what the Tas event record contains, for the proxy check in Phase 3.

In [8]:
ev = pd.read_csv("../labels/events.csv", parse_dates=["date"])
mai = ev[ev.station_code == "MAI"]
print(mai.groupby(["taxon", mai.date.dt.year]).size().unstack(fill_value=0).to_string())
print("closure level exceeded:", int(mai.exceeds_closure_level.sum()), "of", len(mai))

date                   2010  2011  2015  2016  2017  2018  2020  2022  2023  2024  2025  2026
taxon                                                                                        
Alexandrium               0     0     1     2     1     0     3     2     2     0     0     1
Dinophysis                1     0     0     0     0     0     0     0     0     1     0     0
Gymnodinium catenatum     0     0     0     0     0     0     0     0     1     1     3     0
Karenia                   0     0     0     0     0     0     0     0     1     0     0     0
Pseudo-nitzschia          1     1     0     1     1     1     1     0     0     0     1     0
closure level exceeded: 4 of 27


## Findings

1. **Coverage:** about 30% of area-days have a valid chl value (≥5 px), rising from about 20%
   in 2016-2018 (Sentinel-3A only) to about 37% from 2019 (3A + 3B). About 62% of training and
   75% of validation issue dates get a label. Winter coverage is much lower.
2. **`MIN_VALID_PIXELS` stays at 5.** Day-to-day noise falls smoothly as the minimum rises,
   with no clear elbow, so stricter minimums just trade coverage for noise at a steady rate.
   The 7-day forward mean already averages much of the noise away. `n_valid` is kept as a
   quality feature.
3. **Fixed baseline is primary; trailing is a sensitivity analysis.** On validation the two
   are almost the same: positive rate 13.8% vs 13.0%, quarterly median anomaly near 0, and
   no sign of drift through 2023. Validation can't separate them. The fixed baseline keeps 29%
   more training labels (the trailing one has none in 2016 and few in 2017-2018). The
   satellite decline seen at Maria Island (D006) mostly falls in the test years, so both
   labels are reported in Phase 3.
4. **Validation positive rate (~13.8%) is above the 10% design rate.** Training years range
   from 6% to 14%, so 2023 is within normal year-to-year variation. Risk-tier cut-offs are
   chosen on validation anyway.
5. **Drivers:** recent chl anomaly (14-day mean) is the strongest single association, as
   expected (persistence). **Past rain (30 days) is nearly as strong.** That fits runoff
   nutrients, but also coloured dissolved organic matter being misread as chl in estuaries.
   This is a proxy caveat to check in Phase 3.
6. **Harmful-species events at MAI are sparse:** 27 events since 2010, 4 at closure level.
   That is too few to validate the chl proxy statistically in Tasmania; SA and ShellMAP closure
   reasons (Biotoxin News) will be needed.